# TrendPulse - Part 4: prepare the tables

In [1]:
import pandas as pd
import os

COUNTRIES = ["US", "GB", "IN"]

frames = []
for country in COUNTRIES:
    d = pd.read_csv(f"data/{country}videos.csv")
    d["country"] = country
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

df["trending_date"] = pd.to_datetime(df["trending_date"], format="%y.%d.%m")
df["publish_time"] = pd.to_datetime(df["publish_time"], utc=True).dt.tz_localize(None)
df["likes"] = df["likes"].fillna(0).astype("int64")
df["comment_count"] = df["comment_count"].fillna(0).astype("int64")

df = df.drop_duplicates(subset=["video_id", "trending_date", "country"])
df = df[df["video_error_or_removed"] == False].reset_index(drop=True)

print("Clean records:", len(df))

Clean records: 45908


In [2]:
categories = pd.read_csv("data/categories.csv")[["category_id", "category_name"]]

videos = (df.sort_values("trending_date")
            .drop_duplicates("video_id")
            [["video_id", "title", "channel_title", "category_id", "publish_time"]]
            .copy())
videos["publish_time"] = videos["publish_time"].dt.strftime("%Y-%m-%d %H:%M:%S")

records = df[["video_id", "country", "trending_date",
              "views", "likes", "dislikes", "comment_count"]].copy()
records["trending_date"] = records["trending_date"].dt.strftime("%Y-%m-%d")
records.insert(0, "record_id", range(1, len(records) + 1))

print("categories:", len(categories))
print("videos:", len(videos))
print("trending_records:", len(records))

categories: 17
videos: 14688
trending_records: 45908


In [3]:
print("Every video category exists in categories:",
      videos["category_id"].isin(categories["category_id"]).all())
print("Every record's video exists in videos:",
      records["video_id"].isin(videos["video_id"]).all())
print("video_id is unique in videos:", videos["video_id"].is_unique)

Every video category exists in categories: True
Every record's video exists in videos: True
video_id is unique in videos: True


In [4]:
import shutil
os.makedirs("data_out", exist_ok=True)

categories.to_csv("data_out/categories.csv", index=False)
videos.to_csv("data_out/videos.csv", index=False)
records.to_csv("data_out/trending_records.csv", index=False)

shutil.make_archive("data_out", "zip", "data_out")
print(os.listdir("data_out"))

['videos.csv', 'categories.csv', 'trending_records.csv']


In [5]:
import pandas as pd

FOLDER = "data_out"   # the folder with your three CSV files

categories = pd.read_csv(f"{FOLDER}/categories.csv")
videos = pd.read_csv(f"{FOLDER}/videos.csv")
records = pd.read_csv(f"{FOLDER}/trending_records.csv")

flat = records.merge(videos[["video_id", "title", "category_id"]], on="video_id", how="left")
flat["trending_date"] = pd.to_datetime(flat["trending_date"])
flat = flat[["record_id", "video_id", "title", "category_id", "country",
             "trending_date", "views", "likes", "dislikes", "comment_count"]]

with pd.ExcelWriter("trendpulse_workbook.xlsx", engine="openpyxl",
                    datetime_format="yyyy-mm-dd") as writer:
    flat.to_excel(writer, sheet_name="Records", index=False)
    categories.to_excel(writer, sheet_name="Lookup", index=False)

print(flat.shape)

from google.colab import files
files.download("trendpulse_workbook.xlsx")

(45908, 10)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>